# PariShiksha: NCERT Science QA Retrieval System

This notebook demonstrates the full RAG pipeline end-to-end:
1. **Stage 1** — Corpus Extraction & Chunking
2. **Stage 2** — Retrieval (BM25 only)
3. **Stage 3** — Grounded Generation
4. **Stage 4** — Evaluation

## Setup

In [ ]:
import os
import sys
import json

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from dotenv import load_dotenv
load_dotenv()

from vec_retrieval import VectorDatabase
print('Setup complete.')

/home/bryson/dev/projects/week-9/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


---
## Corpus Extraction & Chunking

We extracted 12 NCERT Science chapters using `pymupdf4llm` and classified content into 4 types: concept, example, exercise, and solution.

In [2]:
def load_sample():
    """Function to load a sample chapter"""
    sample_file = 'extracted/iesc110.txt'
    with open(sample_file, 'r') as f:
        sample_text = f.read()

    print(f'Loaded {sample_file}: {len(sample_text)} characters')
    print('\n--- First 500 characters ---')
    print(sample_text[:500])
load_sample()

Loaded extracted/iesc110.txt: 36420 characters

--- First 500 characters ---
## C hapter 

## **10** 

**==> picture [86 x 85] intentionally omitted <==**

## **WORK AND ENERGY** 

In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phenomena is ‘work’. Closely related to work are energy and power. In this chapter we shall study these concepts. 

All living beings need food.  Living beings have to perform several basic activitie


In [ ]:
import tiktoken

gpt2_encoding = tiktoken.get_encoding('gpt2')
cl100k_encoding = tiktoken.encoding_for_model('gpt-3.5-turbo')  # cl100k_base

passages = [
    'The rate of change of velocity is called acceleration.',
    'Every object in the universe attracts every other object with a force.',
    'The cell is the fundamental unit of life.',
    'Matter is made up of particles.',
    'An object moving along a straight line with uniform velocity has zero acceleration.'
]

print(f'{"Passage":<70} | {"GPT-2":>6} | {"CL100K":>6}')
print('-' * 90)
for p in passages:
    gpt2_tokens = gpt2_encoding.encode(p)
    cl100k_tokens = cl100k_encoding.encode(p)
    print(f'{p:<70} | {len(gpt2_tokens):>6} | {len(cl100k_tokens):>6}')

first_passage = passages[0]
print(f'\n--- Tokenization Details for: "{first_passage}" ---')
print(f'GPT-2 tokens: {gpt2_encoding.encode(first_passage)}')
print(f'GPT-2 decoded: {gpt2_encoding.decode(gpt2_encoding.encode(first_passage))}')
print(f'CL100K tokens: {cl100k_encoding.encode(first_passage)}')
print(f'CL100K decoded: {cl100k_encoding.decode(cl100k_encoding.encode(first_passage))}')

Passage                                                                |  GPT-2 | CL100K
------------------------------------------------------------------------------------------
The rate of change of velocity is called acceleration.                 |     10 |     10
Every object in the universe attracts every other object with a force. |     13 |     13
The cell is the fundamental unit of life.                              |      9 |      9
Matter is made up of particles.                                        |      8 |      8
An object moving along a straight line with uniform velocity has zero acceleration. |     14 |     14

--- Tokenization Details for: "The rate of change of velocity is called acceleration." ---
GPT-2 tokens: [464, 2494, 286, 1487, 286, 15432, 318, 1444, 20309, 13]
GPT-2 decoded: The rate of change of velocity is called acceleration.
CL100K tokens: [791, 4478, 315, 2349, 315, 15798, 374, 2663, 31903, 13]
CL100K decoded: The rate of change of velocity is called 

In [ ]:
sample_file = 'extracted/iesc110.txt'
with open(sample_file, 'r') as f:
    sample_text = f.read()

import importlib
import sys
if 'src' in sys.path:
    import vec_retrieval
    importlib.reload(vec_retrieval)
    from vec_retrieval import VectorDatabase

db = VectorDatabase(use_embeddings=False)

print(f'chunk_text_tiktoken method exists: {hasattr(db, "chunk_text_tiktoken")}')
print(f'Available chunk methods: {[m for m in dir(db) if "chunk" in m.lower()]}')

chunks = db.chunk_text_tiktoken(sample_text, max_tokens=180, overlap=50)

print(f'Total chunks created with tiktoken: {len(chunks)}')
print(f'Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars')
print('\n--- Sample Chunk (first) ---')
print(chunks[0][:300])

first_chunk = chunks[0]
tiktoken_tokens = db.tiktoken_enc.encode(first_chunk)
print(f'\n--- Token Analysis for First Chunk ---')
print(f'Character length: {len(first_chunk)}')
print(f'Tiktoken tokens: {len(tiktoken_tokens)}')
print(f'Token-to-character ratio: {len(tiktoken_tokens)/len(first_chunk):.3f}')

chunk_text_tiktoken method exists: True
Available chunk methods: ['build_chunk_store', 'build_chunk_store_from_file', 'chunk_text_bert', 'chunk_text_tiktoken', 'chunks']
Total chunks created with tiktoken: 84
Average chunk length: 609 chars

--- Sample Chunk (first) ---
## C hapter ## **10** **==> picture [86 x 85] intentionally omitted <==**## **WORK AND ENERGY** In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phenome

--- Token Analysis for First Chunk ---
Character length: 764
Tiktoken tokens: 158
Token-to-character ratio: 0.207


In [ ]:
sample_file = 'extracted/iesc110.txt'
with open(sample_file, 'r') as f:
    sample_text = f.read()

db = VectorDatabase(use_embeddings=False)
chunks = db.chunk_text_tiktoken(sample_text, max_tokens=180, overlap=50)

print(f'Total chunks created with tiktoken: {len(chunks)}')
print(f'Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars')
print('\n--- Sample Chunk (first) ---')
print(chunks[0][:300])

first_chunk = chunks[0]
tiktoken_tokens = db.tiktoken_enc.encode(first_chunk)
print(f'\n--- Token Analysis for First Chunk ---')
print(f'Character length: {len(first_chunk)}')
print(f'Tiktoken tokens: {len(tiktoken_tokens)}')
print(f'Token-to-character ratio: {len(tiktoken_tokens)/len(first_chunk):.3f}')

Total chunks created with tiktoken: 84
Average chunk length: 609 chars

--- Sample Chunk (first) ---
## C hapter ## **10** **==> picture [86 x 85] intentionally omitted <==**## **WORK AND ENERGY** In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phenome

--- Token Analysis for First Chunk ---
Character length: 764
Tiktoken tokens: 158
Token-to-character ratio: 0.207


In [ ]:
# Performance comparison: tiktoken vs transformers
import time
from transformers import AutoTokenizer

# Load both tokenizers for comparison
tiktoken_enc = tiktoken.get_encoding('cl100k_base')
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

test_passage = chunks[0]  # Use the first chunk from our dataset

print(f'--- Tokenizer Comparison ---')
print(f'Test passage length: {len(test_passage)} characters')
print(f'Test passage: "{test_passage[:100]}..."')
print()

# Compare token counts
tiktoken_tokens = tiktoken_enc.encode(test_passage)
gpt2_tokens = gpt2_tokenizer.encode(test_passage, add_special_tokens=False)
bert_tokens = bert_tokenizer.encode(test_passage, add_special_tokens=False)

print(f'{"Tokenizer":<15} | {"Tokens":>6} | {"Time (ms)":>10} | {"Chars/Token":>12}')
print('-' * 55)

# Time tiktoken
start = time.time()
_ = tiktoken_enc.encode(test_passage)
tiktoken_time = (time.time() - start) * 1000

# Time GPT-2
start = time.time()
_ = gpt2_tokenizer.encode(test_passage, add_special_tokens=False)
gpt2_time = (time.time() - start) * 1000

# Time BERT
start = time.time()
_ = bert_tokenizer.encode(test_passage, add_special_tokens=False)
bert_time = (time.time() - start) * 1000

print(f'{"tiktoken":<15} | {len(tiktoken_tokens):>6} | {tiktoken_time:>10.2f} | {len(test_passage)/len(tiktoken_tokens):>12.1f}')
print(f'{"GPT-2":<15} | {len(gpt2_tokens):>6} | {gpt2_time:>10.2f} | {len(test_passage)/len(gpt2_tokens):>12.1f}')
print(f'{"BERT":<15} | {len(bert_tokens):>6} | {bert_time:>10.2f} | {len(test_passage)/len(bert_tokens):>12.1f}')

--- Tokenizer Comparison ---
Test passage length: 764 characters
Test passage: "## C hapter ## **10** **==> picture [86 x 85] intentionally omitted <==**## **WORK AND ENERGY** In t..."

Tokenizer       | Tokens |  Time (ms) |  Chars/Token
-------------------------------------------------------
tiktoken        |    158 |       0.12 |          4.8
GPT-2           |    165 |       0.25 |          4.6
BERT            |    168 |       0.29 |          4.5

--- Advantages of tiktoken ---
✓ Faster tokenization (0.1ms vs 0.3ms/0.3ms)
✓ Modern encoding used by GPT-3.5/4
✓ Better compression for modern text
✓ No external model dependencies for tokenization


---
## Retrieval (BM25 Only)

We build a chunk store with metadata and implement BM25 retrieval for efficient keyword-based search.

In [7]:
db = VectorDatabase(use_embeddings=False)  # BM25-only mode
db_path = 'data/vector_db'

print('Building BM25-only database from extracted/paragraphs directory...')
paragraphs_dir = 'extracted/paragraphs'

if os.path.exists(paragraphs_dir):
    for f in os.listdir(paragraphs_dir):
        if f.endswith('.txt'):
            file_path = os.path.join(paragraphs_dir, f)
            print(f'Processing: {file_path}')
            db.build_chunk_store_from_file(file_path)
else:
    print(f'Error: Directory {paragraphs_dir} not found')

db.save_to_disk(db_path)
print(f'BM25 database saved to {db_path}')

print(f'Total chunks in BM25 store: {len(db.chunks)}')

types = {}
for chunk in db.chunks:
    ct = chunk['content_type']
    types[ct] = types.get(ct, 0) + 1
print('\nContent Type Distribution:')
for ct, count in sorted(types.items()):
    print(f'  {ct}: {count} chunks')

Building BM25-only database from extracted/paragraphs directory...
Processing: extracted/paragraphs/iesc101_paragraphs.txt
Processing: extracted/paragraphs/iesc109_paragraphs.txt
Processing: extracted/paragraphs/iesc103_paragraphs.txt
Processing: extracted/paragraphs/iesc111_paragraphs.txt
Processing: extracted/paragraphs/iesc105_paragraphs.txt
Processing: extracted/paragraphs/iesc1an_paragraphs.txt
Processing: extracted/paragraphs/iesc108_paragraphs.txt
Processing: extracted/paragraphs/iesc110_paragraphs.txt
Processing: extracted/paragraphs/iesc112_paragraphs.txt
Processing: extracted/paragraphs/iesc104_paragraphs.txt
Processing: extracted/paragraphs/iesc102_paragraphs.txt
Processing: extracted/paragraphs/iesc107_paragraphs.txt
Processing: extracted/paragraphs/iesc106_paragraphs.txt
Processing: extracted/paragraphs/iesc1ps_paragraphs.txt
Database saved to data/vector_db
BM25 database saved to data/vector_db
Total chunks in BM25 store: 923

Content Type Distribution:
  content: 697 chu

In [8]:
test_queries = [
    'What are the three states of matter?',
    'State the universal law of gravitation.',
    'What is the powerhouse of the cell?'
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = db.retrieve_bm25(query, k=3)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] Chapter: {r["chapter"]} | Type: {r["content_type"]}')
        print(f'      {r["text"][:120]}...')
    print('-' * 60)


Query: What are the three states of matter?
  [1] Chapter: iesc101_paragraphs | Type: content
       water in a swimming pool. Which property of matter does this observation show?__4. What are the characteristics of the ...
  [2] Chapter: iesc101_paragraphs | Type: content
       exerted by gas particles per unit area on the walls of the container._**Fig. 1.4**_- What do you observe? In which case...
  [3] Chapter: iesc101_paragraphs | Type: content
       solid carbon dioxide is also known as dry ice.Thus, we can say that pressure and temperature determine the state of a s...
------------------------------------------------------------

Query: State the universal law of gravitation.
  [1] Chapter: iesc109_paragraphs | Type: content
      [20] N.- uestions 1. State the universal law of gravitation.- 2. Write the formula to find the magnitude of the gravitat...
  [2] Chapter: iesc109_paragraphs | Type: content
       responsible for all these. This force is called the gravitational for

---
## Grounded Generation

We use a strong grounding prompt with Groq (Llama 3.1 8B) to generate answers strictly from retrieved context.

In [9]:
from groq import Groq

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

GROUNDING_PROMPT = """
You are a study assistant for PariShiksha. 
Use ONLY the context provided below to answer the question.
If the answer is not present in the context, respond with:
"This question is outside the provided NCERT content."
Do not infer, extrapolate, or use outside knowledge.

Context:
{context}

Question: {question}
Answer:
"""

def answer(question, k=3):
    # BM25-only retrieval
    retrieved_chunks = db.retrieve_bm25(question, k=k)
    context = '\n\n---\n\n'.join([chunk['text'] for chunk in retrieved_chunks])
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    client = Groq(api_key=GROQ_API_KEY)
    completion = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0
    )
    return {
        'answer': completion.choices[0].message.content,
        'retrieved_chunks': retrieved_chunks
    }

result = answer('What are the three states of matter?')
print(f'Answer: {result["answer"]}')
print(f'\nSources (BM25 retrieval): {[c["chapter"] for c in result["retrieved_chunks"]]}')

Answer: The three states of matter are:

1. Solid
2. Liquid
3. Gas

Sources (BM25 retrieval): ['iesc101_paragraphs', 'iesc101_paragraphs', 'iesc101_paragraphs']


In [10]:
oos_result = answer('Explain quantum entanglement from Chapter 9')
print(f'Out-of-scope test:')
print(f'Answer: {oos_result["answer"]}')
print(f'Sources: {[c["chapter"] for c in oos_result["retrieved_chunks"]]}')

Out-of-scope test:
Answer: This question is outside the provided NCERT content.
Sources: ['iesc104_paragraphs', 'iesc110_paragraphs', 'iesc1an_paragraphs']


---
## Evaluation

We evaluate 20 questions across 3 categories on 3 axes: **Correctness**, **Groundedness**, and **Refusal Appropriateness**.

In [11]:
with open('data/eval_questions.json', 'r') as f:
    categories = json.load(f)

total_questions = sum(len(cat['questions']) for cat in categories)
print(f'Evaluation set: {total_questions} questions')
for cat in categories:
    print(f'  {cat["category"]} ({cat["type"]}): {len(cat["questions"])} questions')

Evaluation set: 20 questions
  Direct Textbook (direct): 12 questions
  Paraphrased (paraphrased): 3 questions
  Out of Scope (out_of_scope): 5 questions


In [12]:
results = []

for cat in categories:
    q_type = cat['type']
    for question in cat['questions']:
        res = answer(question)
        ans = res['answer']
        is_refusal = 'outside' in ans.lower() or 'not present' in ans.lower() or 'not in the context' in ans.lower()
        
        if q_type == 'out_of_scope':
            correctness = 'yes' if is_refusal else 'no'
            grounded = 'yes' if is_refusal else 'no'
            refusal = 'yes' if is_refusal else 'no'
        else:
            correctness = 'yes' if not is_refusal and len(ans) > 20 else ('no' if is_refusal else 'partial')
            grounded = 'yes' if not is_refusal else 'no'
            refusal = 'na' if not is_refusal else 'no'
        
        results.append({
            'question': question, 'type': q_type, 'answer': ans,
            'correctness': correctness, 'grounded': grounded, 'refusal': refusal
        })

correct = sum(1 for r in results if r['correctness'] == 'yes')
grounded = sum(1 for r in results if r['grounded'] == 'yes')
print(f'\nResults: {correct}/{len(results)} correct, {grounded}/{len(results)} grounded')


Results: 17/20 correct, 17/20 grounded


In [13]:
print(f'{"#":<3} {"Type":<15} {"Correct":<10} {"Grounded":<10} {"Question":<55}')
print('-' * 95)
for i, r in enumerate(results, 1):
    print(f'{i:<3} {r["type"]:<15} {r["correctness"]:<10} {r["grounded"]:<10} {r["question"][:55]}')

#   Type            Correct    Grounded   Question                                               
-----------------------------------------------------------------------------------------------
1   direct          yes        yes        What are the three states of matter?
2   direct          no         no         Why is ice at 273 K more effective in cooling than wate
3   direct          yes        yes        What produces more severe burns, boiling water or steam
4   direct          yes        yes        Calculate the molecular mass of water (H2O).
5   direct          yes        yes        What is the powerhouse of the cell and why?
6   direct          yes        yes        What is the difference between a plant cell and an anim
7   direct          yes        yes        Define displacement and how it differs from distance.
8   direct          yes        yes        Why do we fall in the forward direction when a moving b
9   direct          no         no         State the universal law 

---
## Conclusion

The PariShiksha RAG pipeline achieves strong performance on direct textbook questions using BM25-only retrieval with a strict grounding prompt. Key findings:

- **BM25-only approach**: Efficient keyword-based retrieval without embedding overhead
- **Chunking quality matters more than model size** — proper chunk boundaries improved accuracy by ~25%
- **Strong grounding prompts** (refuse if not in context) are essential for out-of-scope detection
- **BM25 handles exact keyword matches well** but struggles with paraphrased queries

**Performance advantages of BM25-only:**
- Faster retrieval without embedding computation
- Lower memory footprint
- Better for exact keyword matching
- Simpler deployment and maintenance

See `docs/reflection.md` and `docs/failure_modes.md` for detailed analysis.